In [ ]:
CLASS_NAMES = ["picaras", "chokosoda", "oreo"]

In [ ]:

# ── 1. FUNCIONES DE PÉRDIDA modelo 2──────────────────────────────

def iou_metric_2(y_true, y_pred):
    """Métrica IoU para monitorear en TensorBoard (no diferenciable, solo lectura)."""
    x1g, y1g, x2g, y2g = tf.unstack(y_true, axis=-1)
    x1p, y1p, x2p, y2p = tf.unstack(y_pred, axis=-1)
    xi1 = tf.maximum(x1g, x1p);  yi1 = tf.maximum(y1g, y1p)
    xi2 = tf.minimum(x2g, x2p);  yi2 = tf.minimum(y2g, y2p)
    inter = tf.maximum(xi2 - xi1, 0.0) * tf.maximum(yi2 - yi1, 0.0)
    area_g = tf.maximum((x2g - x1g) * (y2g - y1g), 0.0)
    area_p = tf.maximum((x2p - x1p) * (y2p - y1p), 0.0)
    union  = area_g + area_p - inter + 1e-7
    return tf.reduce_mean(inter / union)


def giou_loss_2(y_true, y_pred):
    """
    GIoU con protección contra coordenadas invertidas.
    Las predicciones se ordenan (x1<x2, y1<y2) antes de calcular.
    """
    # --- Garantizar x1<x2, y1<y2 en predicciones ---
    x1p_raw, y1p_raw, x2p_raw, y2p_raw = tf.unstack(y_pred, axis=-1)
    x1p = tf.minimum(x1p_raw, x2p_raw)
    y1p = tf.minimum(y1p_raw, y2p_raw)
    x2p = tf.maximum(x1p_raw, x2p_raw)
    y2p = tf.maximum(y1p_raw, y2p_raw)

    x1g, y1g, x2g, y2g = tf.unstack(y_true, axis=-1)

    # Intersección
    xi1 = tf.maximum(x1g, x1p);  yi1 = tf.maximum(y1g, y1p)
    xi2 = tf.minimum(x2g, x2p);  yi2 = tf.minimum(y2g, y2p)
    inter = tf.maximum(xi2 - xi1, 0.0) * tf.maximum(yi2 - yi1, 0.0)

    area_g = (x2g - x1g) * (y2g - y1g)
    area_p = (x2p - x1p) * (y2p - y1p)
    union  = area_g + area_p - inter + 1e-7
    iou    = inter / union

    # Caja envolvente (penalización GIoU)
    xc1 = tf.minimum(x1g, x1p);  yc1 = tf.minimum(y1g, y1p)
    xc2 = tf.maximum(x2g, x2p);  yc2 = tf.maximum(y2g, y2p)
    area_c = (xc2 - xc1) * (yc2 - yc1) + 1e-7

    giou = iou - (area_c - union) / area_c
    return tf.reduce_mean(1.0 - giou)


def mse_loss_2(y_true, y_pred):
    """MSE puro como etapa de calentamiento."""
    return tf.reduce_mean(tf.square(y_true - y_pred))


def bbox_combined_loss_2(y_true, y_pred):
    mse  = mse_loss_2(y_true, y_pred)
    giou = giou_loss_2(y_true, y_pred)

    # Penalizar si el área predicha es muy diferente al área real
    w_true = y_true[:, 2] - y_true[:, 0]
    h_true = y_true[:, 3] - y_true[:, 1]
    w_pred = y_pred[:, 2] - y_pred[:, 0]
    h_pred = y_pred[:, 3] - y_pred[:, 1]
    size_loss = tf.reduce_mean(tf.square(w_true - w_pred) + tf.square(h_true - h_pred))

    return mse + giou + 0.5 * size_loss


In [ ]:

# ── 1. Cargar el modelo 1──────────────────────────────────
modelo1 = tf.keras.models.load_model(
    'mejor_modelo_mobilenet.keras',
    custom_objects={
        'bbox_combined_loss': bbox_combined_loss,
        'mse_loss': mse_loss,
        'giou_loss': giou_loss,
        'iou_metric': iou_metric
    }
)





In [ ]:
import tensorflow as tf
import numpy as np

# ── 1. Cargar el modelo 2 ──────────────────────────────────
modelo2 = tf.keras.models.load_model(
    'mejor_modelo_mobilenet_rotacion.keras',
    custom_objects={
        'bbox_combined_loss': bbox_combined_loss_2,
        'mse_loss': mse_loss_2,
        'giou_loss': giou_loss_2,
        'iou_metric': iou_metric_2
    }
)


